In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2
import seaborn as sns


# ---------------------------------------------------------
# Core Statistical Backtesting Engine
# ---------------------------------------------------------
def run_var_backtest(
    actual_losses, var_series, conf_level=0.99, model_name="VaR Model"
):
  # Align valid dates (drop NaNs)
  valid_mask = var_series.notna() & actual_losses.notna()
  losses = actual_losses[valid_mask]
  vars_val = var_series[valid_mask]

  N = len(losses)
  p0 = 1.0 - conf_level  # Expected failure rate (0.01 for 99%)

  # Identify exceptions
  exceptions = (losses > vars_val).astype(int)
  x = exceptions.sum()
  p_hat = x / N if N > 0 else 0.0

  # 1. Kupiec POF Test
  if x == 0:
    lr_pof = -2 * N * np.log(1 - p0)
  elif x == N:
    lr_pof = -2 * N * np.log(p0)
  else:
    log_num = (N - x) * np.log(1 - p0) + x * np.log(p0)
    log_den = (N - x) * np.log(1 - p_hat) + x * np.log(p_hat)
    lr_pof = -2 * (log_num - log_den)

  p_val_pof = 1.0 - chi2.cdf(lr_pof, df=1)

  # 2. Christoffersen Independence Test
  exc_curr = exceptions.values[1:]
  exc_prev = exceptions.values[:-1]

  n00 = np.sum((exc_prev == 0) & (exc_curr == 0))
  n01 = np.sum((exc_prev == 0) & (exc_curr == 1))
  n10 = np.sum((exc_prev == 1) & (exc_curr == 0))
  n11 = np.sum((exc_prev == 1) & (exc_curr == 1))

  pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0
  pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
  pi = (
      (n01 + n11) / (n00 + n01 + n10 + n11)
      if (n00 + n01 + n10 + n11) > 0
      else 0.0
  )

  def safe_log(val):
    return np.log(val) if val > 0 else 0.0

  log_L0 = (n00 + n10) * safe_log(1 - pi) + (n01 + n11) * safe_log(pi)
  log_L1 = (
      n00 * safe_log(1 - pi0)
      + n01 * safe_log(pi0)
      + n10 * safe_log(1 - pi1)
      + n11 * safe_log(pi1)
  )

  lr_ind = -2 * (log_L0 - log_L1)
  p_val_ind = 1.0 - chi2.cdf(lr_ind, df=1)

  # 3. Christoffersen Conditional Coverage Test
  lr_cc = lr_pof + lr_ind
  p_val_cc = 1.0 - chi2.cdf(lr_cc, df=2)

  # 4. Basel Traffic Light Classification (Scaled to 250 days equivalent)
  scaled_x = (x / N) * 250 if N > 0 else 0.0
  if scaled_x <= 4.99:
    zone = "Green"
    status = "PASS (Model Valid)"
  elif scaled_x <= 9.99:
    zone = "Yellow"
    status = "WARNING (Under Monitoring)"
  else:
    zone = "Red"
    status = "FAIL (Reject Model)"

  return {
      "Model": model_name,
      "N_Days": N,
      "Exceptions": x,
      "Expected_Exceptions": round(N * p0, 1),
      "Exception_Ratio": f"{p_hat:.2%}",
      "Kupiec_POF_Stat": round(lr_pof, 4),
      "Kupiec_POF_PValue": round(p_val_pof, 4),
      "POF_Conclusion": "PASS" if p_val_pof > 0.05 else "FAIL",
      "Christoffersen_IND_Stat": round(lr_ind, 4),
      "Christoffersen_IND_PValue": round(p_val_ind, 4),
      "IND_Conclusion": "PASS" if p_val_ind > 0.05 else "FAIL",
      "Conditional_Coverage_PValue": round(p_val_cc, 4),
      "CC_Conclusion": "PASS" if p_val_cc > 0.05 else "FAIL",
      "Basel_Zone": zone,
      "Basel_Status": status,
      "Exceptions_Series": exceptions,
      "Losses_Evaluated": losses,  # Aligned losses series
  }


def run_phase_8():
  print("==================================================")
  print("      RUNNING PHASE 8: VaR BACKTESTING ENGINE     ")
  print("==================================================")

  # 1. Load Data
  p3_file = "portfolio_phase3_output(10).csv"
  if not os.path.exists(p3_file):
    raise FileNotFoundError(f"'{p3_file}' not found! Please run Phase 3 first.")

  df_p3 = pd.read_csv(p3_file, parse_dates=["Date"], index_col="Date")
  pnl = df_p3["Daily_PnL"]
  actual_losses = -pnl
  dates = df_p3.index

  ESTIMATION_WINDOW = 750  # Out-of-sample split
  conf_level = 0.99
  alpha = 1.0 - conf_level

  print(f" Backtesting Out-Of-Sample Period (Initial Window: 750 Days)")

  # Generate 750-Day Rolling Historical VaR
  raw_var_750 = pd.Series(index=dates, dtype=float)
  for i in range(ESTIMATION_WINDOW, len(dates)):
    win_pnl = pnl.iloc[i - ESTIMATION_WINDOW : i]
    raw_var_750.iloc[i] = -np.percentile(win_pnl, alpha * 100)

  # Load previous phase VaR series if available for comparison
  models_to_test = {
      "Historical VaR (750d)": raw_var_750,
  }

  if os.path.exists("ewma_scaled_var_summary_phase5(10).csv"):
    p5_df = pd.read_csv(
        "ewma_scaled_var_summary_phase5(10).csv",
        parse_dates=["Date"],
        index_col="Date",
    )
    models_to_test["EWMA Volatility-Scaled VaR"] = p5_df["Scaled_99_VaR"]

  if os.path.exists("garch_scaled_var_summary_phase6(10).csv"):
    p6_df = pd.read_csv(
        "garch_scaled_var_summary_phase6(10).csv",
        parse_dates=["Date"],
        index_col="Date",
    )
    models_to_test["GARCH(1,1)-Scaled VaR"] = p6_df["GARCH_Scaled_99_VaR"]

  if os.path.exists("gjr_garch_scaled_var_summary_phase7(10).csv"):
    p7_df = pd.read_csv(
        "gjr_garch_scaled_var_summary_phase7(10).csv",
        parse_dates=["Date"],
        index_col="Date",
    )
    models_to_test["GJR-GARCH-Scaled VaR"] = p7_df["GJR_GARCH_Scaled_99_VaR"]

  # 2. Run Backtest across all available VaR models
  summary_list = []
  results_dict = {}

  for name, var_ser in models_to_test.items():
    res = run_var_backtest(
        actual_losses, var_ser, conf_level=conf_level, model_name=name
    )
    results_dict[name] = res

    # Store for summary table
    row = {
        k: v
        for k, v in res.items()
        if k not in ["Exceptions_Series", "Losses_Evaluated"]
    }
    summary_list.append(row)

  backtest_summary = pd.DataFrame(summary_list)
  backtest_summary.to_csv("var_backtest_summary_phase8(10).csv", index=False)
  print(" [✓] Saved Backtest Summary -> var_backtest_summary_phase8(10).csv")

  print("\n--- BACKTESTING RESULTS TABLE ---")
  print(
      backtest_summary[[
          "Model",
          "Exceptions",
          "Expected_Exceptions",
          "Exception_Ratio",
          "Kupiec_POF_PValue",
          "POF_Conclusion",
          "Christoffersen_IND_PValue",
          "Basel_Zone",
          "Basel_Status",
      ]].to_string(index=False)
  )

  # ---------------------------------------------------------
  # 3. Generate Charts (BUG FIXED HERE)
  # ---------------------------------------------------------
  sns.set_theme(style="whitegrid")

  # CHART 1: Out-of-Sample Losses vs VaR Threshold & Exceptions
  primary_var = raw_var_750
  primary_res = results_dict["Historical VaR (750d)"]

  # FIX: Use the aligned losses and reindexed mask to avoid IndexingError
  eval_losses = primary_res["Losses_Evaluated"]
  exceptions_mask = primary_res["Exceptions_Series"] == 1

  plt.figure(figsize=(12, 6))
  plt.plot(
      actual_losses.index,
      actual_losses,
      color="silver",
      alpha=0.6,
      lw=0.8,
      label="Actual Portfolio Dollar Loss ($)",
  )
  plt.plot(
      primary_var.index,
      primary_var,
      color="navy",
      lw=1.5,
      label="99% Historical VaR (750d Out-of-Sample)",
  )

  # Scatter exception breaches using aligned losses
  breach_vals = eval_losses[exceptions_mask]
  plt.scatter(
      breach_vals.index,
      breach_vals,
      color="red",
      s=40,
      zorder=5,
      label=f"VaR Exceptions ({len(breach_vals)} Breaches)",
  )

  plt.title("Chart 1: Out-of-Sample VaR Backtesting (Actual Losses vs. 99% VaR)")
  plt.xlabel("Date")
  plt.ylabel("Dollar Loss ($)")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart1_var_backtest_timeseries(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart1_var_backtest_timeseries(10).png")

  # CHART 2: Basel Traffic Light Zone Comparison Across Models
  models_names = backtest_summary["Model"]
  exc_counts = backtest_summary["Exceptions"]
  colors = []

  for z in backtest_summary["Basel_Zone"]:
    if z == "Green":
      colors.append("mediumseagreen")
    elif z == "Yellow":
      colors.append("gold")
    else:
      colors.append("crimson")

  plt.figure(figsize=(10, 5))
  bars = plt.bar(
      models_names, exc_counts, color=colors, width=0.5, edgecolor="black"
  )

  plt.axhline(
      4, color="green", linestyle="--", lw=1.5, label="Green Zone Threshold (≤4)"
  )
  plt.axhline(
      9, color="orange", linestyle="--", lw=1.5, label="Yellow Zone Threshold (≤9)"
  )

  for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 0.3,
        int(yval),
        ha="center",
        va="bottom",
        fontweight="bold",
    )

  plt.title("Chart 2: Basel Traffic Light Backtest Exception Comparison")
  plt.ylabel("Number of VaR Exceptions")
  plt.xticks(rotation=15)
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart2_basel_traffic_light_comparison(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart2_basel_traffic_light_comparison(10).png")

  print("\n==================================================")
  print("          PHASE 8 COMPLETED SUCCESSFULLY!         ")
  print("==================================================")


if __name__ == "__main__":
  run_phase_8()